# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a ranking / scoring task, not a plain yes/no classification. The decision it improves is "out of thousands of pages, which ones does a content reviewer look at first?" — that's an ordering problem, not a single-page verdict. Under the hood it's built on a binary classifier (is_declining_label), because a probability from that classifier is exactly what makes a good priority score: pages get sorted by predicted decline risk, and the reviewer works down the list until capacity runs out. So scoring/ranking is the task type; classification is the mechanism producing the score — matching the starter pipeline itself (03_train_model.py outputs a probability, 04_evaluate_and_export.py turns it into a ranked queue).


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is a proxy label, not a fully observed future outcome:
is_declining_label = (trend_direction == "down")
trend_direction is itself computed from trend_pct, a rule applied to the current 90-day window — so this label describes where a page already stands, not what happens to it next. That's why trend_direction and trend_pct can never be model features (they'd leak the answer into the input). This ML-03 framing uses the starter proxy as-is, matching Lane 2's default. A stronger capstone version would use a genuinely observed future outcome — e.g. prior 90-day features → decline over the next 30 days.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50: of the top 50 pages the ranked queue puts first, how many are actually declining by the label above? This beats accuracy or ROC-AUC as the metric to defend because it matches how the output gets used — a content team has limited review capacity, so what matters is whether the top of the list is worth the reviewer's time. "Good" means clearly beating the hand-rule baseline, which the reference pipeline already reports at Precision@50 = 0.240 (outputs/model_report.md).

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one pseudonymized content page (content_id), restricted to pages with real search demand (impressions_90d > 0) that are old enough to have a trend (content_age_days >= 90), deduplicated by content_id — the same eligibility rule 01_prepare_features.py applies. On the 30,000-row starter CSV this yields the eligible subset actually scored by the pipeline.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/albijanashala/ML1/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(URL)

# Eligibility rule, matching scripts/01_prepare_features.py
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
eligible = eligible.drop_duplicates(subset="content_id")

print(f"Rows loaded : {len(df):,}")
print(f"Eligible    : {len(eligible):,}")
print(f"Unique pages: {eligible['content_id'].nunique():,}")
print(f"Clients     : {eligible['client_id'].nunique()}")

eligible[["content_id", "client_id", "content_type", "impressions_90d", "trend_direction"]].head()

The eligibility rule keeps all 30,000 rows on this starter slice — nothing is
dropped. Rows and unique content_id both equal 30,000, so one row is one page.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The baseline (02_baseline_score.py) is one fixed linear formula — four signals, fixed weights (0.40 * visibility + 0.30 * freshness_risk + 0.25 * position_opportunity + 0.05 * depth_gap) — applied identically to every page regardless of content type or how signals interact. Real decline isn't linear/additive like that: a page can look fine on any single signal while still being at risk once position, freshness, engagement, and content type combine. A tree-based model learns those interactions automatically. Evidence already committed in this repo: baseline rule scores 0.240 at Precision@50, random forest scores 0.740 — roughly a 3x lift, same eligible pages, same label, same client-holdout split.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.